# 실습 1: 협업 필터링 파이프라인 Overview

이 실습은 MovieLens 데이터를 불러와 User-based CF와 Item-based CF를 **처음부터 끝까지** 한 번에 체험하는 오버뷰(Overview) 실습입니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/4주차/lab_01_pipeline.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 환경 준비: 라이브러리 임포트

In [ ]:
import pandas as pd
import torch

torch.manual_seed(42)

## 2. 데이터 다운로드

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip -O ml-100k.zip
!unzip -q -o ml-100k.zip
print("다운로드 완료")

## 3. User-Item 평점 행렬 구성

In [ ]:
df = pd.read_csv(
    'ml-100k/u.data', sep='\t', header=None,
    names=['user_id', 'item_id', 'rating', 'timestamp']
)
movies = pd.read_csv(
    'ml-100k/u.item', sep='|', header=None, encoding='latin-1',
    usecols=[0, 1], names=['item_id', 'title']
)
movie_names = dict(zip(movies['item_id'], movies['title']))

n_users = df['user_id'].max()
n_items = df['item_id'].max()

ratings = torch.zeros(n_users, n_items)
for row in df.itertuples():
    ratings[row.user_id - 1, row.item_id - 1] = row.rating

print(f"행렬 크기: {ratings.shape}  (사용자 × 영화)")

## 4. 코사인 유사도 계산

In [ ]:
norms = torch.norm(ratings, dim=1, keepdim=True).clamp(min=1e-8)
normalized = ratings / norms
user_sim = torch.mm(normalized, normalized.T)  # (943, 943)

print(f"사용자 유사도 행렬 크기: {user_sim.shape}")
print(f"사용자 0과 사용자 1의 유사도: {user_sim[0, 1].item():.4f}")

## 5. User-based CF

In [ ]:
def recommend_user_based(user_idx, ratings, user_sim, top_n_neighbors=20, top_n_items=10):
    sim_scores = user_sim[user_idx].clone()
    sim_scores[user_idx] = -1

    top_neighbors = torch.topk(sim_scores, top_n_neighbors).indices

    unrated_mask = (ratings[user_idx] == 0)
    neighbor_ratings = ratings[top_neighbors]           # (top_n_neighbors, n_items)
    weights = sim_scores[top_neighbors].unsqueeze(1)    # (top_n_neighbors, 1)

    scores = (neighbor_ratings * weights).sum(dim=0)    # (n_items,)
    scores[~unrated_mask] = -float('inf')

    return torch.topk(scores, top_n_items).indices.tolist()

## 6. 추천 생성

In [ ]:
user_idx = 0
ub_rec_items = recommend_user_based(user_idx, ratings, user_sim)

print(f"사용자 {user_idx + 1}번의 User-based CF 추천 영화 Top 10:")
for rank, item_idx in enumerate(ub_rec_items, 1):
    print(f"  {rank:2d}. {movie_names.get(item_idx + 1, 'Unknown')}")

## 7. Item-based CF와의 비교

In [ ]:
item_norms = torch.norm(ratings.T, dim=1, keepdim=True).clamp(min=1e-8)
item_normalized = ratings.T / item_norms
item_sim = torch.mm(item_normalized, item_normalized.T)  # (1682, 1682)

def recommend_item_based(user_idx, ratings, item_sim, top_n=10):
    user_ratings = ratings[user_idx]
    unrated_mask = (user_ratings == 0)
    scores = torch.mv(item_sim, user_ratings)
    scores[~unrated_mask] = -float('inf')
    return torch.topk(scores, top_n).indices.tolist()

ib_rec_items = recommend_item_based(user_idx, ratings, item_sim)

print(f"사용자 {user_idx + 1}번의 Item-based CF 추천 영화 Top 10:")
for rank, item_idx in enumerate(ib_rec_items, 1):
    print(f"  {rank:2d}. {movie_names.get(item_idx + 1, 'Unknown')}")

ub_set = set(ub_rec_items)
ib_set = set(ib_rec_items)
print(f"\n겹치는 영화 ({len(ub_set & ib_set)}편):")
for item_idx in ub_set & ib_set:
    print(f"  - {movie_names.get(item_idx + 1, 'Unknown')}")

## 8. 학습 결과 정리
- MovieLens ml-100k 평점 데이터를 User-Item 행렬로 변환하고 User-based CF를 실행했습니다.
- User-based CF와 Item-based CF의 추천 결과를 나란히 비교했습니다.